# 03 Training Science — Regularisation

**Status:** Complete

**Learning outcome:** Understand L1, L2, dropout, and early stopping as regularisation techniques — when each helps, how they differ mathematically, and their effect on the train-val gap.

## Why This Matters

Without regularisation, neural networks memorise training data — achieving 100% train accuracy while failing on unseen data. Regularisation closes the train-val gap by penalising model complexity. Each technique has a different mechanism: L2 shrinks weights, L1 sparsifies them, dropout creates an implicit ensemble, and early stopping limits effective model capacity.

In [1]:
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset, random_split
from sklearn.datasets import load_digits

torch.manual_seed(42)
np.random.seed(42)
print("Libraries loaded.")

Libraries loaded.


In [2]:
# Data setup — use small training set to amplify overfitting
digits = load_digits()
X = torch.tensor(digits.data / 16.0, dtype=torch.float32)
y = torch.tensor(digits.target, dtype=torch.long)

dataset = TensorDataset(X, y)
n = len(dataset)
n_train = 200  # deliberately small to provoke overfitting
n_val = n - n_train
train_ds, val_ds = random_split(dataset, [n_train, n_val],
                                 generator=torch.Generator().manual_seed(42))

train_dl = DataLoader(train_ds, batch_size=64, shuffle=True)
val_dl = DataLoader(val_ds, batch_size=256)
print(f"Train: {n_train}, Val: {n_val} (small train → overfitting expected)")


def train_model(model, optimizer, epochs=150):
    """Train and return history."""
    criterion = nn.CrossEntropyLoss()
    hist = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}

    for epoch in range(epochs):
        model.train()
        for xb, yb in train_dl:
            optimizer.zero_grad()
            loss = criterion(model(xb), yb)
            loss.backward()
            optimizer.step()

        # Evaluate
        model.eval()
        with torch.no_grad():
            # Train
            all_preds, all_loss = [], 0.0
            for xb, yb in train_dl:
                logits = model(xb)
                all_loss += criterion(logits, yb).item() * len(xb)
                all_preds.extend((logits.argmax(1) == yb).tolist())
            hist['train_loss'].append(all_loss / n_train)
            hist['train_acc'].append(sum(all_preds) / len(all_preds))
            # Val
            all_preds, all_loss = [], 0.0
            for xb, yb in val_dl:
                logits = model(xb)
                all_loss += criterion(logits, yb).item() * len(xb)
                all_preds.extend((logits.argmax(1) == yb).tolist())
            hist['val_loss'].append(all_loss / n_val)
            hist['val_acc'].append(sum(all_preds) / len(all_preds))

    return hist

Train: 200, Val: 1597 (small train → overfitting expected)


---
**Understanding Overfitting: What It Looks Like and Why It Happens**

**Plain language:** Imagine a student who memorises every answer in the textbook word-for-word but can't solve a problem they haven't seen before. That's overfitting — the model has learned the training examples by heart instead of learning the underlying patterns that generalise to new data.

**Building intuition:** When a model has far more parameters than training examples, it has enough capacity to assign a unique "rule" to every data point — essentially building a lookup table. Training accuracy climbs to 100%, but validation accuracy plateaus or even drops. The gap between train and val performance is called the *generalisation gap*, and it's the primary diagnostic for overfitting. Reducing this gap without sacrificing too much training performance is the goal of regularisation.

**Formal statement:** A model $f_\theta$ overfits when the empirical risk $\hat{R}(\theta) = \frac{1}{n}\sum_{i=1}^{n}\ell(f_\theta(x_i), y_i)$ is minimised on the training set while the population risk $R(\theta) = \mathbb{E}_{(x,y)\sim\mathcal{D}}[\ell(f_\theta(x), y)]$ remains large. The generalisation gap $R(\theta) - \hat{R}(\theta)$ grows when model complexity exceeds what the data can constrain.

---

## Experiment: No Reg vs L2 (Weight Decay) vs Dropout

In [3]:
def make_model(dropout=0.0):
    layers = [nn.Linear(64, 256), nn.ReLU()]
    if dropout > 0:
        layers.append(nn.Dropout(dropout))
    layers += [nn.Linear(256, 256), nn.ReLU()]
    if dropout > 0:
        layers.append(nn.Dropout(dropout))
    layers.append(nn.Linear(256, 10))
    return nn.Sequential(*layers)

results = {}

# 1. No regularisation
torch.manual_seed(42)
model_noreg = make_model()
results['No reg'] = train_model(model_noreg, optim.Adam(model_noreg.parameters(), lr=1e-3))

# 2. L2 / weight decay
torch.manual_seed(42)
model_l2 = make_model()
results['L2 (wd=0.01)'] = train_model(model_l2, optim.AdamW(model_l2.parameters(), lr=1e-3, weight_decay=0.01))

# 3. Dropout
torch.manual_seed(42)
model_drop = make_model(dropout=0.3)
results['Dropout 0.3'] = train_model(model_drop, optim.Adam(model_drop.parameters(), lr=1e-3))

# 4. Both
torch.manual_seed(42)
model_both = make_model(dropout=0.3)
results['L2 + Dropout'] = train_model(model_both, optim.AdamW(model_both.parameters(), lr=1e-3, weight_decay=0.01))

for name, h in results.items():
    gap = h['train_acc'][-1] - h['val_acc'][-1]
    print(f"{name:>15s}: train_acc={h['train_acc'][-1]:.2%}, val_acc={h['val_acc'][-1]:.2%}, gap={gap:.2%}")

         No reg: train_acc=100.00%, val_acc=92.49%, gap=7.51%
   L2 (wd=0.01): train_acc=100.00%, val_acc=92.49%, gap=7.51%
    Dropout 0.3: train_acc=100.00%, val_acc=93.36%, gap=6.64%
   L2 + Dropout: train_acc=100.00%, val_acc=93.24%, gap=6.76%


---
**Understanding L2 vs L1: Weight Shrinkage vs Sparsity**

**Plain language:** Think of L2 regularisation as a tax that charges proportionally more on large weights — like a progressive tax that discourages extremes but lets everyone keep something. L1 regularisation is a flat tax — the same penalty regardless of size — which drives the smallest weights all the way to zero, effectively eliminating them.

**Building intuition:** L2 adds $\lambda \sum w_i^2$ to the loss. Because the penalty is quadratic, large weights are penalised much more heavily than small ones, so all weights shrink toward zero but rarely reach it. L1 adds $\lambda \sum |w_i|$, a linear penalty. The gradient of $|w|$ is constant ($\pm 1$), so it pushes every weight toward zero with equal force — small weights get eliminated entirely. This makes L1 a natural *feature selector*: unimportant features get exactly zero weight.

**Formal statement:** L2-regularised loss: $\mathcal{L}_{\text{L2}} = \mathcal{L}_{\text{data}} + \lambda \|\mathbf{w}\|_2^2$. L1-regularised loss: $\mathcal{L}_{\text{L1}} = \mathcal{L}_{\text{data}} + \lambda \|\mathbf{w}\|_1$. The L2 gradient contribution is $2\lambda w_i$ (proportional to weight magnitude); the L1 gradient contribution is $\lambda \cdot \text{sign}(w_i)$ (constant magnitude). In practice, AdamW implements *decoupled weight decay* $w \leftarrow (1 - \lambda)w - \alpha \nabla \mathcal{L}$, which is equivalent to L2 for SGD but differs for adaptive optimisers.

---

In [4]:
# Visualise train vs val loss curves
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
colors = {'No reg': 'red', 'L2 (wd=0.01)': 'blue', 'Dropout 0.3': 'green', 'L2 + Dropout': 'purple'}

for name, h in results.items():
    c = colors[name]
    axes[0].plot(h['train_loss'], '-', color=c, alpha=0.7, label=f'{name} (train)')
    axes[0].plot(h['val_loss'], '--', color=c, alpha=0.7, label=f'{name} (val)')

axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
axes[0].set_title('Train vs Val Loss'); axes[0].legend(fontsize=7)

for name, h in results.items():
    gap = [t - v for t, v in zip(h['train_acc'], h['val_acc'])]
    axes[1].plot(gap, color=colors[name], label=name)

axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Train - Val Accuracy')
axes[1].set_title('Generalisation Gap'); axes[1].legend()
axes[1].axhline(0, color='black', ls=':', alpha=0.3)

plt.suptitle('Regularisation Effect on Overfitting', fontweight='bold')
plt.tight_layout()
plt.savefig('../assets/regularisation.png', dpi=100)
plt.show()

# Assertion: regularised models should have smaller gap
noreg_gap = results['No reg']['train_acc'][-1] - results['No reg']['val_acc'][-1]
both_gap = results['L2 + Dropout']['train_acc'][-1] - results['L2 + Dropout']['val_acc'][-1]
assert both_gap <= noreg_gap + 0.05, "Regularisation should reduce or maintain the gap"
print(f"\nGeneralisation gap: No reg = {noreg_gap:.2%}, L2+Dropout = {both_gap:.2%} ✓")


Generalisation gap: No reg = 7.51%, L2+Dropout = 6.76% ✓


/var/folders/sy/gz5gl6d91cbfwd2z85r62rbc0000gn/T/ipykernel_70061/1456177567.py:24: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


---
**Understanding Dropout as Implicit Ensemble**

**Plain language:** Imagine a team where, during every practice session, random members are told to sit out. The remaining players must learn to cover for absent teammates, so everyone becomes more versatile. On game day, the full team plays together and their combined, redundant skills make them stronger than a team that always practised with the same lineup.

**Building intuition:** During training, dropout randomly sets each neuron's output to zero with probability $p$. This prevents *co-adaptation* — the tendency for specific neurons to rely on each other in brittle ways. Each training step effectively trains a different "thin" subnetwork. At test time, all neurons are active, but their outputs are scaled by $(1-p)$ to compensate for the increased number of active units. The result is approximately equivalent to averaging the predictions of all $2^n$ possible subnetworks — an exponentially large ensemble from a single model.

**Formal statement:** During training, each hidden unit $h_i$ is multiplied by a Bernoulli mask $m_i \sim \text{Bernoulli}(1-p)$: $\tilde{h}_i = m_i \cdot h_i$. At test time, $\tilde{h}_i = (1-p) \cdot h_i$. This is equivalent (in expectation) to the geometric mean of the predictions of all $2^d$ subnetworks, where $d$ is the number of units subject to dropout (Baldi & Sadowski, 2013).

---

In [ ]:
# Weight Distribution Histograms — No Reg vs L2 vs Dropout
wd_fig, wd_axes = plt.subplots(1, 3, figsize=(14, 4), sharey=True)
wd_models = {'No Reg': model_noreg, 'L2 (wd=0.01)': model_l2, 'Dropout 0.3': model_drop}
wd_colors = {'No Reg': 'red', 'L2 (wd=0.01)': 'blue', 'Dropout 0.3': 'green'}

for ax, (name, model) in zip(wd_axes, wd_models.items()):
    all_weights = []
    for p in model.parameters():
        if p.dim() >= 2:  # weight matrices only, skip biases
            all_weights.append(p.detach().cpu().numpy().flatten())
    all_weights = np.concatenate(all_weights)
    ax.hist(all_weights, bins=80, color=wd_colors[name], alpha=0.7, edgecolor='black', linewidth=0.3)
    ax.set_title(name, fontweight='bold')
    ax.set_xlabel('Weight value')
    ax.axvline(0, color='black', ls=':', alpha=0.5)
    std = np.std(all_weights)
    ax.text(0.95, 0.95, f'std={std:.4f}', transform=ax.transAxes,
            ha='right', va='top', fontsize=9, bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

wd_axes[0].set_ylabel('Count')
wd_fig.suptitle('Weight Distributions After Training', fontweight='bold')
wd_fig.tight_layout()
wd_fig.savefig('../assets/weight_distributions.png', dpi=100)
plt.show()
print("Weight distribution histograms saved to ../assets/weight_distributions.png")

In [ ]:
# Dropout Mask Visualization — 5 forward passes through same input
torch.manual_seed(42)
dm_fig, dm_ax = plt.subplots(figsize=(10, 4))

# Create a dropout layer and record masks across 5 forward passes
dropout_layer = nn.Dropout(p=0.3)
dropout_layer.train()  # must be in train mode to actually drop

# Use a fixed input of ones so the mask is visible (1 = active, 0 = dropped)
dummy_input = torch.ones(1, 128)  # 128 neurons wide
masks = []
for i in range(5):
    out = dropout_layer(dummy_input)
    # Dropout scales by 1/(1-p), so active neurons show ~1.43, dropped show 0
    mask = (out > 0).float().squeeze().numpy()
    masks.append(mask)

# Reshape to 8x16 for each pass for a compact heatmap
mask_grid = np.array(masks)[:, :128].reshape(5, 8, 16)
# Stack vertically: 5 passes, each 8x16
combined = mask_grid.reshape(5 * 8, 16)

dm_ax.imshow(combined, cmap='RdYlGn', aspect='auto', interpolation='nearest')

# Add horizontal lines to separate the 5 passes
for i in range(1, 5):
    dm_ax.axhline(i * 8 - 0.5, color='black', linewidth=2)

dm_ax.set_yticks([4, 12, 20, 28, 36])
dm_ax.set_yticklabels(['Pass 1', 'Pass 2', 'Pass 3', 'Pass 4', 'Pass 5'])
dm_ax.set_xlabel('Neuron index (within 8x16 block)')
dm_ax.set_title('Dropout Masks: Active (green) vs Dropped (red) Across 5 Forward Passes',
                fontweight='bold')

# Add colorbar
from matplotlib.colors import ListedColormap
cmap_binary = ListedColormap(['#d32f2f', '#4caf50'])
dm_ax2 = dm_fig.add_axes([0.92, 0.15, 0.02, 0.7])
cb = plt.colorbar(plt.cm.ScalarMappable(cmap=cmap_binary), cax=dm_ax2)
cb.set_ticks([0.25, 0.75])
cb.set_ticklabels(['Dropped', 'Active'])

dm_fig.subplots_adjust(right=0.9)
dm_fig.savefig('../assets/dropout_masks.png', dpi=100, bbox_inches='tight')
plt.show()
print("Dropout mask visualization saved to ../assets/dropout_masks.png")

## Structured Interpretation

1. **No regularisation** leads to overfitting: train accuracy approaches 100% while val accuracy plateaus or declines. The widening gap between train and val loss is the hallmark of memorisation.

2. **L2 / weight decay** penalises large weights, keeping the effective model capacity smaller. In AdamW, weight decay is decoupled from the adaptive learning rate, giving uniform shrinkage.

3. **Dropout** randomly zeros activations during training, forcing the network to learn redundant representations. At test time, all neurons are active (with scaled weights), creating an implicit ensemble.

4. **Combining L2 + dropout** typically gives the best regularisation. They complement each other: L2 shrinks weights globally, dropout prevents co-adaptation of specific neurons.

5. **Early stopping** (demonstrated in the previous notebook) is another form of regularisation — it limits the number of gradient updates, preventing the model from fitting noise in the training data.

6. **For MNEMOSYNE**: We'll use AdamW (built-in weight decay) + dropout in our surrogate model, with early stopping on validation loss as the stopping criterion.